<a href="https://colab.research.google.com/github/gseetharami352005/CSA6102/blob/main/exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import re

def detect_phishing(sender, subject, body):
    """Detects potential phishing emails based on suspicious indicators."""
    suspicious_score = 0
    indicators = []

    # --- Sender Analysis ---
    # 1. Mismatched domain (simple check: if sender name doesn't match email domain well)
    # This is a very basic check; real-world scenarios need more sophisticated domain analysis.
    sender_name_match = re.search(r'"?([^"]+)"? <([^>]+)>', sender)
    sender_address = sender
    sender_display_name = None
    if sender_name_match:
        sender_display_name = sender_name_match.group(1).lower()
        sender_address = sender_name_match.group(2).lower()
    else:
        sender_address = sender.lower()

    domain = sender_address.split('@')[-1]
    if sender_display_name and not (sender_display_name in domain or domain in sender_display_name):
        # A very crude check, will trigger for many legitimate emails (e.g., 'Google <noreply@google.com>')
        # More refined logic needed for production.
        suspicious_score += 1
        indicators.append(f"Sender: Display name '{sender_display_name}' does not clearly match domain '{domain}'")

    # 2. Suspicious sender patterns (e.g., long strings of numbers, unusual characters)
    if re.search(r'\d{5,}|[\W_]{3,}', sender_address.split('@')[0]): # 5+ digits or 3+ non-alphanumeric chars in local part
        suspicious_score += 1
        indicators.append("Sender: Unusual characters or long numbers in email address local part.")

    # --- Subject Analysis ---
    subject_lower = subject.lower()
    suspicious_subject_keywords = [
        "urgent", "action required", "account suspended", "verify your account",
        "security alert", "payment failed", "invoice due", "password reset",
        "unexpected activity", "your bank", "prize winner", "lottery",
        "delivery failure", "update information"
    ]
    for keyword in suspicious_subject_keywords:
        if keyword in subject_lower:
            suspicious_score += 1
            indicators.append(f"Subject: Contains suspicious keyword: '{keyword}'")

    # Check for excessive capitalization
    if sum(1 for c in subject if c.isupper()) > len(subject) / 2 and len(subject) > 10:
        suspicious_score += 1
        indicators.append("Subject: Excessive capitalization.")

    # --- Body Analysis ---
    body_lower = body.lower()

    # 1. Generic greetings
    generic_greetings = ["dear customer", "dear user", "valued client", "hi there"]
    if any(greeting in body_lower for greeting in generic_greetings):
        suspicious_score += 1
        indicators.append("Body: Uses a generic greeting.")

    # 2. Urgent call to action or threats
    urgent_phrases = [
        "click here immediately", "your account will be closed", "failure to comply",
        "if you do not respond", "verify your details now", "avoid suspension"
    ]
    for phrase in urgent_phrases:
        if phrase in body_lower:
            suspicious_score += 1
            indicators.append(f"Body: Contains urgent call to action/threat: '{phrase}'")

    # 3. Suspicious links (presence of a link that doesn't clearly match sender domain)
    # This is a very complex check in reality. Here, we just detect *any* HTTP/HTTPS link.
    urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', body)
    if urls:
        suspicious_score += 1
        indicators.append(f"Body: Contains {len(urls)} external URL(s).")
        # A more advanced check would extract the domain from the URL and compare it to the sender's domain.

    # 4. Typos and grammatical errors (very basic check, hard to do accurately without NLP)
    # We'll just look for a few common phishing typos as an example
    common_typos = {"paypall": "paypal", "amaz0n": "amazon", "microsof": "microsoft"}
    for typo, correct in common_typos.items():
        if typo in body_lower:
            suspicious_score += 1
            indicators.append(f"Body: Contains potential typo: '{typo}' (should be '{correct}')")

    return suspicious_score, indicators

# --- Demonstration ---

# Example 1: Legitimate-looking email
email1_sender = "Bank of America <noreply@bankofamerica.com>"
email1_subject = "Your Monthly Statement Available"
email1_body = "Dear Customer, your monthly statement is now available. Log in to your account to view it."

print("\n--- Analyzing Email 1 (Legitimate-looking) ---")
score, indicators = detect_phishing(email1_sender, email1_subject, email1_body)
print(f"Phishing Score: {score}")
if indicators:
    print("Indicators found:")
    for indicator in indicators:
        print(f"- {indicator}")
else:
    print("No significant phishing indicators found.")

# Example 2: Phishing email (Urgent, generic, suspicious link)
email2_sender = "PayPal Customer Service <support@paypall-verify.net>"
email2_subject = "URGENT: Your PayPal Account Has Been Limited!"
email2_body = (
    "Dear Valued Member,\n\n"
    "We have detected unusual activity on your account. To avoid suspension, please click the link below and verify your details immediately.\n"
    "Click Here To Secure Your Account: https://secure-paypall.net/verify/login\n\n"
    "Failure to comply will result in permanent account closure. Thank you for your cooperation.\n"
    "The PayPal Team"
)

print("\n--- Analyzing Email 2 (Phishing Attempt) ---")
score, indicators = detect_phishing(email2_sender, email2_subject, email2_body)
print(f"Phishing Score: {score}")
if indicators:
    print("Indicators found:")
    for indicator in indicators:
        print(f"- {indicator}")
else:
    print("No significant phishing indicators found.")

# Example 3: Another phishing example (Generic sender, suspicious subject/body)
email3_sender = "Admin Support <admin_help123@generic-email.xyz>"
email3_subject = "ATTENTION: MANDATORY EMAIL UPDATE REQUIRED IMMEDIATELY"
email3_body = (
    "Dear User,\n\n"
    "Our records indicate that your email account requires an immediate update. If you do not update your information within 24 hours, your account will be deleted.\n"
    "Proceed to update: http://update.your-mail.net/login.php\n\n"
    "This is a security measure. Thank you."
)

print("\n--- Analyzing Email 3 (Another Phishing Attempt) ---")
score, indicators = detect_phishing(email3_sender, email3_subject, email3_body)
print(f"Phishing Score: {score}")
if indicators:
    print("Indicators found:")
    for indicator in indicators:
        print(f"- {indicator}")
else:
    print("No significant phishing indicators found.")


--- Analyzing Email 1 (Legitimate-looking) ---
Phishing Score: 2
Indicators found:
- Sender: Display name 'bank of america' does not clearly match domain 'bankofamerica.com'
- Body: Uses a generic greeting.

--- Analyzing Email 2 (Phishing Attempt) ---
Phishing Score: 6
Indicators found:
- Sender: Display name 'paypal customer service' does not clearly match domain 'paypall-verify.net'
- Subject: Contains suspicious keyword: 'urgent'
- Body: Contains urgent call to action/threat: 'failure to comply'
- Body: Contains urgent call to action/threat: 'avoid suspension'
- Body: Contains 1 external URL(s).
- Body: Contains potential typo: 'paypall' (should be 'paypal')

--- Analyzing Email 3 (Another Phishing Attempt) ---
Phishing Score: 4
Indicators found:
- Sender: Display name 'admin support' does not clearly match domain 'generic-email.xyz'
- Subject: Excessive capitalization.
- Body: Uses a generic greeting.
- Body: Contains 1 external URL(s).
